# Reload

In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
import random
import datetime

from time_slot import TimeSlot
from locations import Locations
from truck import *
from freight import Freight

#Imports for the AI algorithms
import numpy as np
import gymnasium as gym
from gymnasium import spaces

import os
import time

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.ppo import MlpPolicy

#Imports for the imitation learning
from imitation.data.wrappers import RolloutInfoWrapper
from imitation.util.util import make_vec_env

from stable_baselines3.common.evaluation import evaluate_policy

from imitation.data import rollout

from imitation.algorithms import bc

from imitation.algorithms import preference_comparisons
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from imitation.policies.base import FeedForward32Policy, NormalizeFeaturesExtractor
from imitation.rewards.reward_wrapper import RewardVecEnvWrapper

from imitation.data.types import Trajectory, Transitions
import pickle

from imitation.algorithms.adversarial.gail import GAIL
from imitation.algorithms.adversarial.airl import AIRL
from imitation.rewards.reward_nets import BasicShapedRewardNet

import tempfile
from imitation.algorithms.dagger import SimpleDAggerTrainer

#Explainability
import sklearn
import sklearn.datasets
import sklearn.ensemble
import lime
import lime.lime_tabular
from __future__ import print_function
import pandas as pd
from sklearn.utils import Bunch
import torch

# Parameters

In [3]:
number_of_trucks = 125
number_of_freight = 250

# Time Frame

In [4]:
start_time = datetime.datetime.now() 
print("start_time =", start_time)

#Time frame of 1 week
number_of_days = 7

delta_time = datetime.timedelta(days=1) 
delta_time_seconds_day = 24 * 60 * 60 #Seconds in a day
delta_time_seconds_week = number_of_days * 24 * 60 * 60 #Seconds in a Week

time_frame = []

for i in range (number_of_days):
    end_time = start_time + delta_time
    
    time_frame += [start_time]
    start_time = end_time

#Return start_time back to normal
start_time -= datetime.timedelta(seconds=delta_time_seconds_week)
print("start_time =", start_time)
    
print("time_frame =", time_frame, end = '\n\n')

days_of_the_week = []
for i in range(number_of_days):
    days_of_the_week.append(i)

start_time = 2025-10-01 12:20:23.498191
start_time = 2025-10-01 12:20:23.498191
time_frame = [datetime.datetime(2025, 10, 1, 12, 20, 23, 498191), datetime.datetime(2025, 10, 2, 12, 20, 23, 498191), datetime.datetime(2025, 10, 3, 12, 20, 23, 498191), datetime.datetime(2025, 10, 4, 12, 20, 23, 498191), datetime.datetime(2025, 10, 5, 12, 20, 23, 498191), datetime.datetime(2025, 10, 6, 12, 20, 23, 498191), datetime.datetime(2025, 10, 7, 12, 20, 23, 498191)]



# Locations

In [5]:
locations = Locations.locations

# Trucks

In [6]:
def generateRandomTrucks(number_of_trucks):
    trucks = []
    
    #Full list of haulier companies at at https://www.imt-ip.pt/sites/IMTT/Portugues/TransportesRodoviarios/EmpresasLicenciadas/Documents/Empresas_Mercadorias_OrdEmp.pdf
    companies = ["CARKARMA", "CASTRO E MERINO TRANSPORTES", "DESTINOS VIBRANTES, UNIPESSOAL", "DROPLOG", "ESTORILMUDA - TRANSPORTES", "MAIN ROAD CONSTRUCTIONS", "SILVA & AZEVEDO", "TRANSPORTES JOAQUIM MONTEIRO", "TRANSPORTES MALHADA", "TRANSPORTES RAIO", "WINDY TRANSPORT-SUCURSAL EM PORTUGAL"]
    
    truck_types = ["20' Standard - Steel", "40' Standard - Steel", "40' High - Steel", "45' High - Steel", "20' Standard Reefer", "40' High Cube Standard Reefer", "40' High Cube Magnum Reefer", "40' High Cube StarCare", "40' High Cube Super Freezer", "20' Open Top", "40' Open Top High Cube", "40' High Flat Rack Collapsible"]

    for current_id in range(number_of_trucks):
        company = random.choice(companies)
        location = random.choice(locations)

        availability = {}
        
        number_of_time_slots = random.randint(3, 7) #We are considering the time frame of 1 week
        time_ids = random.sample(days_of_the_week, k = number_of_time_slots)

        for time_id in time_ids:
            #from https://stackoverflow.com/questions/553303/how-to-generate-a-random-date-between-two-other-dates?page=1&tab=scoredesc#tab-top
            random_second_start = random.randrange(delta_time_seconds_day) #Random time between the two dates
            new_time_slot_start = time_frame[time_id] + datetime.timedelta(seconds=random_second_start)

            random_second_end = random.randrange(delta_time_seconds_day) #Random time between the two dates
            new_time_slot_end = time_frame[time_id] + datetime.timedelta(seconds=random_second_end)

            if random_second_start < random_second_end: #Start has to come before the end
                new_time_slot = TimeSlot(time_id, new_time_slot_start, new_time_slot_end)
            else: #Start doesn't come before the end, so switch places
                new_time_slot = TimeSlot(time_id, new_time_slot_end, new_time_slot_start)

            availability[time_id] = new_time_slot

        #emission rate in gCO2/t-km; between 58.3 and 307.2 according to https://theicct.org/publication/co2-emissions-from-trucks-in-the-eu-an-analysis-of-the-heavy-duty-co2-standards-baseline-data/
        emission_rate = round(random.uniform(58.3, 307.3), 1)
        
        #Choose type of truck
        #truck_type = random.choice(truck_types)
        truck_type = random.choices(truck_types, cum_weights = [1, 2, 3, 4, 4.8, 5.6, 6.4, 7.2, 8, 9.33, 10.66, 11.99], k = 1)[0]
        
        if (truck_type == "20' Standard - Steel"):
            truck = Dry_20StandardSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' Standard - Steel"):
            truck = Dry_40StandardSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High - Steel"):
            truck = Dry_40HighSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "45' High - Steel"):
            truck = Dry_45HighSteel(current_id, company, location, availability, emission_rate)
        elif (truck_type == "20' Standard Reefer"):
            truck = Refrigerated_20StandardReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Standard Reefer"):
            truck = Refrigerated_40HighCubeStandardReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Magnum Reefer"):
            truck = Refrigerated_40HighCubeMagnumReefer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube StarCare"):
            truck = Refrigerated_40HighCubeStarCare(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Cube Super Freezer"):
            truck = Refrigerated_40HighCubeSuperFreezer(current_id, company, location, availability, emission_rate)
        elif (truck_type == "20' Open Top"):
            truck = Special_20OpenTop(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' Open Top High Cube"):
            truck = Special_40OpenTopHighCube(current_id, company, location, availability, emission_rate)
        elif (truck_type == "40' High Flat Rack Collapsible"):
            truck = Special_40HighFlatRackCollapsible(current_id, company, location, availability, emission_rate)
        
        trucks.append(truck)
        
    return trucks
    


# Freight

In [7]:
def generateRandomFreight(number_of_freight):
    freight_list = []
    
    categories = ["Dry", "Refrigerated", "Special"]

    current_id = 0

    for i in range(number_of_freight):
        category = random.choice(categories)
        
        if (category == "Dry"):
            weight = random.uniform(24, 28.87) #28.87 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 13555) #13556 = length of the container with the most length
            width = random.uniform(2000, 2351) #2352 = width of the widest container
            height = random.uniform(1800, 2697) #2698 = height of the tallest container
        elif (category == "Refrigerated"):
            weight = random.uniform(24, 29.67) #29.67 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 11587) #11588 = length of the container with the most length
            width = random.uniform(2200, 2289) #2290 = width of the widest container
            height = random.uniform(1700, 2504) #2505 = height of the tallest container
        elif (category == "Special"):
            weight = random.uniform(15, 47.3) #47.3 tons = weight capacity of the container with the most capacity considered
            length = random.uniform(3000, 12031) #12032 = length of the container with the most length
            width = random.uniform(2300, 2373) #2374 = width of the widest container
            height = random.uniform(1700, 2652) #2653 = height of the tallest container
        
        origin = random.choice(locations)
        destination = random.choice(locations)
        #Guarantee that origin is different than the destination
        while (origin == destination):
            destination = random.choice(locations)

        random_second_arrival = random.randrange(delta_time_seconds_week) #Random time between the two dates
        new_arrival = start_time + datetime.timedelta(seconds=random_second_arrival)
        
        random_second_deadline = random.randrange(delta_time_seconds_week) #Random time between the two dates
        new_deadline = start_time + datetime.timedelta(seconds=random_second_deadline)
        
        if random_second_deadline > random_second_arrival: #Arrival needs to happen first
            arrival = new_arrival
            deadline = new_deadline
        else:
            arrival = new_deadline
            deadline = new_arrival
        
        freight = Freight(current_id, category, weight, length, width, height, origin, destination, arrival, deadline)
        freight_list.append(freight)
        
        current_id += 1
    
    return freight_list

# Calculate Statistics

In [8]:
def calculateStatistics(trucks):
    total_distance_travelled = 0
    total_distance_travelled_with_no_freight = 0
    total_time_spent = datetime.timedelta(seconds=0)
    total_time_spent_with_no_freight = datetime.timedelta(seconds=0)
    total_grams_of_CO2_emitted = 0

    number_of_freight_delivered = 0

    for truck in trucks:
        timeslots = truck.getTimeSlots()

        for day in days_of_the_week:
            slot = timeslots.get(day)

            if ( (slot) and (not slot.isAvailable()) ):
                total_distance_travelled += slot.getDistanceTravelled()
                total_distance_travelled_with_no_freight += slot.getDistanceTravelledWithNoFreight()
                total_time_spent += slot.getTimeSpent()
                total_time_spent_with_no_freight += slot.getTimeSpentWithNoFreight()
                total_grams_of_CO2_emitted += slot.getGramsOfCO2Emitted()

                number_of_freight_delivered += 1
                
    print("total_distance_travelled:", round (total_distance_travelled, 1), "Km")
    print("total_distance_travelled_with_no_freight:", round(total_distance_travelled_with_no_freight, 1), "Km\n")

    print("total_time_spent:", total_time_spent)
    print("total_time_spent_with_no_freight:", total_time_spent_with_no_freight, end = "\n\n")

    total_grams_of_CO2_emitted = round(total_grams_of_CO2_emitted)
    print("total_grams_of_CO2_emitted:", f"{total_grams_of_CO2_emitted:_}", "grams\n")

    print("number_of_freight_delivered:", number_of_freight_delivered)
    print("percentage of freight delivered:", round( (number_of_freight_delivered / number_of_freight * 100), 1), "%")
    print("\n\n")

# AI Algorithm - Reinforcement Learning

## Create a Custom Environment

In [9]:
class FreightTruckMatchingEnv(gym.Env):
    """Custom Environment that follows gym interface."""

    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 30}

    def __init__(self):
        super().__init__()
        self.action_space = spaces.Discrete(2) #Good time slot or bad time slot

        self.observation_space = spaces.Box(low=-1, high=np.inf,
                                            shape=(3,), dtype=np.float64)

    def step(self, action):            
        ###########################Reward the Agent###########################
        
        self.reward = 0.0
        
        if (self.slot): #The slot was available
            distance_travelled = self.slot["distance_travelled"]
            distance_travelled_with_no_freight = self.slot["distance_travelled_with_no_freight"]
            grams_of_CO2_emitted = self.slot["grams_of_CO2_emitted"]
        
            if (action == 0): #actions: 0 = bad choice, 1 = good choice, according to the agent
                self.bad_slots_truck_list.append( (self.slot, self.trucks[self.truck_ind]) ) #Save it in a list

                if (distance_travelled > 427):
                    self.reward += 10.0
                if (distance_travelled_with_no_freight > 166):
                    self.reward += 10.0
                if (grams_of_CO2_emitted > 1_200_000):
                    self.reward += 10.0

            elif (action == 1): #actions: 0 = bad choice, 1 = good choice, according to the agent
                self.good_slots_truck_list.append( (self.slot, self.trucks[self.truck_ind]) ) #Save it in a list

                if (distance_travelled < 427):
                    self.reward += 10.0
                if(distance_travelled_with_no_freight < 166):
                    self.reward += 10.0
                if (grams_of_CO2_emitted < 1_200_000):
                    self.reward += 10.0
        
            
        ###########################Move to the next truck###########################
        
        self.truck_ind += 1 
        
        if (self.truck_ind >= number_of_trucks): #Every truck covered -> assign freight and move to the next one
            
            ###########################Assign this freight to the timeslot###########################
            
            if (self.good_slots_truck_list): #Check if there is at least one good slot 
                self.slots_truck_list = self.good_slots_truck_list
            elif (self.bad_slots_truck_list): #Else -> Check if there is at least one bad slot 
                self.slots_truck_list = self.bad_slots_truck_list
            
            #Check if there is at least one slot 
            if (self.slots_truck_list): #Slots available -> Let's pick randomly
                best_slot_truck = random.choice(self.slots_truck_list)
                best_truck = best_slot_truck[1]
                best_slot = best_slot_truck[0]

                freight = self.freight_list[self.freight_ind] #Get the current freight
                freight.assignToTruck(best_truck.getTruckId())

                freight_id = self.freight_ind
                freight_destination = freight.getDestination()
                
                timeslot = best_slot["time_slot"]
                distance_travelled = best_slot["distance_travelled"]
                distance_travelled_with_no_freight = best_slot["distance_travelled_with_no_freight"]
                time_spent = best_slot["travel_time"]
                time_spent_with_no_freight = best_slot["time_spent_with_no_freight"]
                grams_of_CO2_emitted = best_slot["grams_of_CO2_emitted"]
                
                timeslot.freightAssignedToTruck(freight_id, distance_travelled, distance_travelled_with_no_freight, time_spent, time_spent_with_no_freight, grams_of_CO2_emitted)

                #Update the truck's location 
                best_truck.updateLocation(freight_destination)
        
            ###########################Move to the next freight###########################
            
            self.slots_truck_list = []
            self.bad_slots_truck_list = []
            self.good_slots_truck_list = []
        
            self.freight_ind += 1
            self.truck_ind = 0
        
            if (self.freight_ind >= number_of_freight): #Every freight covered -> move to the next day
                self.day += 1
                self.freight_ind = 0

                if (self.day >= number_of_days): #We've went through every day
                    self.done = True
                    self.observation = [-1.0, -1.0, -1.0]

                    calculateStatistics(self.trucks)
                
                    self.observation = np.array(self.observation)
                
                    return self.observation, self.reward, self.done, self.truncated, self.info
                
        truck = self.trucks[self.truck_ind] #Get the current truck
        freight = self.freight_list[self.freight_ind] #Get the current freight
        
        if ( not freight.isAssignedToTruck() ):
        
            freight_id = freight.getId()

            freight_category = freight.getCategory()
            freight_weight = freight.getWeight()
            freight_length = freight.getLength()
            freight_width = freight.getWidth()
            freight_height = freight.getHeight()

            freight_arrival = freight.getArrival()
            freight_deadline = freight.getDeadline()
            freight_origin = freight.getOrigin()
            freight_destination = freight.getDestination()
            
            freight_tags = freight.getTags()

            if (truck.containerCanHoldfreight(freight_category, freight_weight, freight_length, freight_width, freight_height, freight_tags)):
                self.slot = truck.timeSlotAvailableForDelivery(self.day, freight_arrival, freight_deadline, freight_origin, freight_destination, freight_weight)
                if (self.slot): #This slot is available
                    
                    #This is the observation
                    self.observation = [ (np.log(self.slot["distance_travelled"] + 1)), (np.log(self.slot["distance_travelled_with_no_freight"] + 1)), (np.log(self.slot["grams_of_CO2_emitted"] + 1)) ]
                    
                else: #This slot is not available
                    self.observation = [-1.0, -1.0, -1.0]
                    self.slot = {}
            else: #The container can't hold the freight
                self.observation = [-1.0, -1.0, -1.0]
                self.slot = {}
        else: #The freight is already assigned
            self.observation = [-1.0, -1.0, -1.0]
            self.slot = {}
                    
        self.observation = np.array(self.observation)
        
        return self.observation, self.reward, self.done, self.truncated, self.info

    def reset(self, seed=None, options=None):
        self.done = False
        self.reward = 0.0
        self.observation = [-1.0, -1.0, -1.0]
        
        self.trucks = generateRandomTrucks(number_of_trucks)
        self.freight_list = generateRandomFreight(number_of_freight)
        
        self.day = 0
        self.freight_ind = 0 #Get the first freight 
        self.truck_ind = 0 #Get The first truck
        
        self.slots_truck_list = []
        self.bad_slots_truck_list = []
        self.good_slots_truck_list = []
                
        ##############################################The first truck and the first freight

        truck = self.trucks[self.truck_ind] #Get the first Truck
        freight = self.freight_list[self.freight_ind] #Get the first freight
        
        freight_id = freight.getId()

        freight_category = freight.getCategory()
        freight_weight = freight.getWeight()
        freight_length = freight.getLength()
        freight_width = freight.getWidth()
        freight_height = freight.getHeight()

        freight_arrival = freight.getArrival()
        freight_deadline = freight.getDeadline()
        freight_origin = freight.getOrigin()
        freight_destination = freight.getDestination()
        
        freight_tags = freight.getTags()
        
        self.slot = {}

        if (truck.containerCanHoldfreight(freight_category, freight_weight, freight_length, freight_width, freight_height, freight_tags)):
            self.slot = truck.timeSlotAvailableForDelivery(self.day, freight_arrival, freight_deadline, freight_origin, freight_destination, freight_weight)
            if (self.slot): #This slot is available
                
                #This is the observation
                self.observation = [ (np.log(self.slot["distance_travelled"] + 1)), (np.log(self.slot["distance_travelled_with_no_freight"] + 1)), (np.log(self.slot["grams_of_CO2_emitted"] + 1)) ]
                
            else: #This slot is not available
                self.observation = [-1.0, -1.0, -1.0]
                self.slot = {}
        else: #The container can't hold the freight
            self.observation = [-1.0, -1.0, -1.0]
            self.slot = {}
                
        
        self.info = {}
        self.truncated = False
        self.observation = np.array(self.observation)
        
        return self.observation, self.info

## Add the environment to the gym registry

In [10]:
gym.register(
    id = "custom/FreightTruckMatching-v1",
    entry_point = FreightTruckMatchingEnv,
    max_episode_steps = number_of_trucks * number_of_freight * number_of_days,
)

## Make sure the environment is valid

In [11]:
#env = FreightTruckMatchingEnv()
env = gym.make("custom/FreightTruckMatching-v1")

# It will check your custom environment and output additional warnings if needed
check_env(env)

## Run it for a couple episodes

In [12]:
#Run it for a couple episodes with random actions to see if there are any errors

episodes = 20

for episode in range(episodes):
    print("\n\n############Episode:", episode, "############\n")
    
    done = False
    obs = env.reset()
    while not done:
        random_action = env.action_space.sample()
        #print("action:", random_action)
        obs, reward, done, truncated, info = env.step(random_action)
        #print('reward:', reward, "\n")



############Episode: 0 ############

total_distance_travelled: 86009.6 Km
total_distance_travelled_with_no_freight: 38195.6 Km

total_time_spent: 36 days, 1:36:00
total_time_spent_with_no_freight: 16 days, 2:02:00

total_grams_of_CO2_emitted: 242_730_050 grams

number_of_freight_delivered: 196
percentage of freight delivered: 78.4 %





############Episode: 1 ############

total_distance_travelled: 95528.3 Km
total_distance_travelled_with_no_freight: 43180.3 Km

total_time_spent: 40 days, 0:14:00
total_time_spent_with_no_freight: 18 days, 5:19:00

total_grams_of_CO2_emitted: 264_554_637 grams

number_of_freight_delivered: 197
percentage of freight delivered: 78.8 %





############Episode: 2 ############

total_distance_travelled: 95957.5 Km
total_distance_travelled_with_no_freight: 41170.7 Km

total_time_spent: 40 days, 1:25:00
total_time_spent_with_no_freight: 17 days, 7:07:00

total_grams_of_CO2_emitted: 287_929_915 grams

number_of_freight_delivered: 205
percentage of freight d

## AI Algorithm - PPO (training)

In [20]:
models_dir = f"models/{int(time.time())}/"
logdir = f"logs/{int(time.time())}/"

if not os.path.exists(models_dir):
    os.makedirs(models_dir)

if not os.path.exists(logdir):
    os.makedirs(logdir)

env.reset()

model = PPO('MlpPolicy', env, verbose=0, tensorboard_log=logdir)

TIMESTEPS = 10000
iters = 0
while True:
    iters += 1
    model.learn(total_timesteps=TIMESTEPS, reset_num_timesteps=False, tb_log_name=f"AI ALgorithm 2")
    model.save(f"{models_dir}/{TIMESTEPS*iters}")

total_distance_travelled: 87117.5 Km
total_distance_travelled_with_no_freight: 36200.2 Km

total_time_spent: 36 days, 21:25:00
total_time_spent_with_no_freight: 15 days, 13:16:00

total_grams_of_CO2_emitted: 257_746_568 grams

number_of_freight_delivered: 207
percentage of freight delivered: 82.8 %



total_distance_travelled: 79293.8 Km
total_distance_travelled_with_no_freight: 30020.8 Km

total_time_spent: 33 days, 21:34:00
total_time_spent_with_no_freight: 13 days, 0:56:00

total_grams_of_CO2_emitted: 258_655_547 grams

number_of_freight_delivered: 199
percentage of freight delivered: 79.6 %



total_distance_travelled: 82963.8 Km
total_distance_travelled_with_no_freight: 28028.8 Km

total_time_spent: 34 days, 22:44:00
total_time_spent_with_no_freight: 12 days, 6:37:00

total_grams_of_CO2_emitted: 282_940_543 grams

number_of_freight_delivered: 201
percentage of freight delivered: 80.4 %



total_distance_travelled: 76331.1 Km
total_distance_travelled_with_no_freight: 25854.0 Km

to

KeyboardInterrupt: 

## Load the Model and Collect the Rollouts

In [12]:
models_dir = "models/1739547997/"
model_path = f"{models_dir}/1350000.zip"

model = PPO.load(model_path, env=env)

y_arr = []
state_arr = []
explanation_arr = []

data_collection_done = False

while(not data_collection_done):
    obs, _ = env.reset()
    
    done = False
    while not done:
        action, _ = model.predict(obs)
        obs, reward, done, truncated, info = env.step(action)
        
        if ( (obs[0] == -1) and (explanation_arr.count("impossible matching")>=20_000) ): #Enough impossible matchings
            continue
        
        y_arr.append(0) #0 means the concept is aligned
        state_arr.append(obs) #state
        
        #Explanation
        if (obs[0] == -1):
            explanation_arr.append("impossible matching")
        else:
            #obs[0] = distance travelled, obs[1] = distance travelled with no freight, obs[2] = grams of CO2 emitted
            distance_travelled_threshold = np.log(427 + 1)
            distance_travelled_with_no_freight_threshold = np.log(166 + 1)
            CO2_threshold = np.log(1_200_000 + 1)
            
            if ( (obs[0] < distance_travelled_threshold) and (obs[1] < distance_travelled_with_no_freight_threshold) and (obs[2] < CO2_threshold) ):
                explanation_arr.append("short distance travelled, short distance travelled with no freight, and low CO2 emissions")
            elif ( (obs[0] >= distance_travelled_threshold) and (obs[1] >= distance_travelled_with_no_freight_threshold) and (obs[2] >= CO2_threshold) ):
                explanation_arr.append("long distance travelled, long distance travelled with no freight, and high CO2 emissions")
                
            elif ( (obs[0] < distance_travelled_threshold) and (obs[1] < distance_travelled_with_no_freight_threshold) and (obs[2] >= CO2_threshold) ):
                explanation_arr.append("short distance travelled and short distance travelled with no freight")
            elif ( (obs[0] < distance_travelled_threshold) and (obs[1] >= distance_travelled_with_no_freight_threshold) and (obs[2] < CO2_threshold) ):
                explanation_arr.append("short distance travelled and low CO2 emissions")
            elif ( (obs[0] >= distance_travelled_threshold) and (obs[1] < distance_travelled_with_no_freight_threshold) and (obs[2] < CO2_threshold) ):
                explanation_arr.append("short distance travelled with no freight and low CO2 emissions")
                
            elif ( (obs[0] >= distance_travelled_threshold) and (obs[1] >= distance_travelled_with_no_freight_threshold) and (obs[2] < CO2_threshold) ):
                explanation_arr.append("long distance travelled and long distance travelled with no freight")
            elif ( (obs[0] >= distance_travelled_threshold) and (obs[1] < distance_travelled_with_no_freight_threshold) and (obs[2] >= CO2_threshold) ):
                explanation_arr.append("long distance travelled and high CO2 emissions")
            elif ( (obs[0] < distance_travelled_threshold) and (obs[1] >= distance_travelled_with_no_freight_threshold) and (obs[2] >= CO2_threshold) ):
                explanation_arr.append("long distance travelled with no freight and high CO2 emissions")
            
            
        if len(y_arr) >= 150_000:
            data_collection_done = True
            break

total_distance_travelled: 75149.2 Km
total_distance_travelled_with_no_freight: 23077.7 Km

total_time_spent: 32 days, 5:42:00
total_time_spent_with_no_freight: 10 days, 7:51:00

total_grams_of_CO2_emitted: 271_301_630 grams

number_of_freight_delivered: 204
percentage of freight delivered: 81.6 %



total_distance_travelled: 71556.6 Km
total_distance_travelled_with_no_freight: 20450.0 Km

total_time_spent: 30 days, 9:58:00
total_time_spent_with_no_freight: 9 days, 0:26:00

total_grams_of_CO2_emitted: 253_610_038 grams

number_of_freight_delivered: 199
percentage of freight delivered: 79.6 %



total_distance_travelled: 69073.0 Km
total_distance_travelled_with_no_freight: 17790.7 Km

total_time_spent: 29 days, 12:20:00
total_time_spent_with_no_freight: 8 days, 2:54:00

total_grams_of_CO2_emitted: 251_193_970 grams

number_of_freight_delivered: 197
percentage of freight delivered: 78.8 %



total_distance_travelled: 77283.0 Km
total_distance_travelled_with_no_freight: 25764.2 Km

total_t

total_distance_travelled: 74917.0 Km
total_distance_travelled_with_no_freight: 22117.6 Km

total_time_spent: 31 days, 18:39:00
total_time_spent_with_no_freight: 9 days, 15:57:00

total_grams_of_CO2_emitted: 250_572_790 grams

number_of_freight_delivered: 201
percentage of freight delivered: 80.4 %



total_distance_travelled: 67970.4 Km
total_distance_travelled_with_no_freight: 18675.8 Km

total_time_spent: 28 days, 23:54:00
total_time_spent_with_no_freight: 8 days, 4:43:00

total_grams_of_CO2_emitted: 249_146_592 grams

number_of_freight_delivered: 198
percentage of freight delivered: 79.2 %



total_distance_travelled: 77145.6 Km
total_distance_travelled_with_no_freight: 24606.5 Km

total_time_spent: 32 days, 15:58:00
total_time_spent_with_no_freight: 10 days, 13:15:00

total_grams_of_CO2_emitted: 252_225_747 grams

number_of_freight_delivered: 204
percentage of freight delivered: 81.6 %



total_distance_travelled: 71946.3 Km
total_distance_travelled_with_no_freight: 22101.1 Km

tot

total_distance_travelled: 73421.1 Km
total_distance_travelled_with_no_freight: 22838.0 Km

total_time_spent: 31 days, 12:41:00
total_time_spent_with_no_freight: 10 days, 3:17:00

total_grams_of_CO2_emitted: 254_707_886 grams

number_of_freight_delivered: 197
percentage of freight delivered: 78.8 %



total_distance_travelled: 71978.9 Km
total_distance_travelled_with_no_freight: 25529.4 Km

total_time_spent: 30 days, 9:32:00
total_time_spent_with_no_freight: 10 days, 22:15:00

total_grams_of_CO2_emitted: 233_067_531 grams

number_of_freight_delivered: 188
percentage of freight delivered: 75.2 %



total_distance_travelled: 69272.3 Km
total_distance_travelled_with_no_freight: 23193.4 Km

total_time_spent: 29 days, 17:50:00
total_time_spent_with_no_freight: 10 days, 5:38:00

total_grams_of_CO2_emitted: 237_524_146 grams

number_of_freight_delivered: 195
percentage of freight delivered: 78.0 %



total_distance_travelled: 82654.5 Km
total_distance_travelled_with_no_freight: 24919.0 Km

tot

total_distance_travelled: 73278.0 Km
total_distance_travelled_with_no_freight: 20679.0 Km

total_time_spent: 31 days, 6:40:00
total_time_spent_with_no_freight: 9 days, 13:57:00

total_grams_of_CO2_emitted: 267_658_069 grams

number_of_freight_delivered: 202
percentage of freight delivered: 80.8 %



total_distance_travelled: 70620.0 Km
total_distance_travelled_with_no_freight: 21194.9 Km

total_time_spent: 30 days, 4:40:00
total_time_spent_with_no_freight: 9 days, 7:01:00

total_grams_of_CO2_emitted: 236_753_798 grams

number_of_freight_delivered: 193
percentage of freight delivered: 77.2 %



total_distance_travelled: 73996.6 Km
total_distance_travelled_with_no_freight: 22431.0 Km

total_time_spent: 31 days, 7:39:00
total_time_spent_with_no_freight: 9 days, 20:24:00

total_grams_of_CO2_emitted: 246_103_995 grams

number_of_freight_delivered: 194
percentage of freight delivered: 77.6 %



total_distance_travelled: 75195.3 Km
total_distance_travelled_with_no_freight: 21874.0 Km

total_t

total_distance_travelled: 63963.6 Km
total_distance_travelled_with_no_freight: 17974.9 Km

total_time_spent: 27 days, 9:01:00
total_time_spent_with_no_freight: 7 days, 20:36:00

total_grams_of_CO2_emitted: 191_863_018 grams

number_of_freight_delivered: 191
percentage of freight delivered: 76.4 %



total_distance_travelled: 72161.4 Km
total_distance_travelled_with_no_freight: 20738.3 Km

total_time_spent: 30 days, 17:09:00
total_time_spent_with_no_freight: 9 days, 3:39:00

total_grams_of_CO2_emitted: 272_839_423 grams

number_of_freight_delivered: 201
percentage of freight delivered: 80.4 %



total_distance_travelled: 82039.0 Km
total_distance_travelled_with_no_freight: 28971.5 Km

total_time_spent: 34 days, 12:44:00
total_time_spent_with_no_freight: 12 days, 7:01:00

total_grams_of_CO2_emitted: 254_908_679 grams

number_of_freight_delivered: 202
percentage of freight delivered: 80.8 %



total_distance_travelled: 78449.9 Km
total_distance_travelled_with_no_freight: 25516.2 Km

total

## Save the dataset

In [13]:
y_arr = np.array(y_arr)
state_arr = np.array(state_arr)
explanation_arr = np.array(explanation_arr)

with open('datasets/data/y_arr_aligned.npy', 'wb') as f:
    np.save(f, y_arr)
    
with open('datasets/data/state_aligned.npy', 'wb') as f:
    np.save(f, state_arr)
    
with open('datasets/data/explanation_aligned.npy', 'wb') as f:
    np.save(f, explanation_arr)

## Load the Dataset to make sure everything is ok

In [14]:
y_arr = np.load("datasets/data/y_arr_aligned.npy", allow_pickle=True)
state_arr = np.load("datasets/data/state_aligned.npy", allow_pickle=True)
explanation_arr = np.load("datasets/data/explanation_aligned.npy", allow_pickle=True)

In [15]:
arr_size = len(explanation_arr)
print(arr_size)

150000


In [16]:
#explanation_arr.count("impossible matching")

unique, counts = np.unique(explanation_arr, return_counts=True)
dict(zip(unique, counts))

{'impossible matching': 20000,
 'long distance travelled and high CO2 emissions': 9337,
 'long distance travelled and long distance travelled with no freight': 28218,
 'long distance travelled with no freight and high CO2 emissions': 996,
 'long distance travelled, long distance travelled with no freight, and high CO2 emissions': 29895,
 'short distance travelled and low CO2 emissions': 15263,
 'short distance travelled and short distance travelled with no freight': 11883,
 'short distance travelled with no freight and low CO2 emissions': 2525,
 'short distance travelled, short distance travelled with no freight, and low CO2 emissions': 31883}

## Create the Misaligned Dataset

In [17]:
z = 8 #number of misaligned concepts

### y_arr_misaligned

In [18]:
y_arr_misaligned = []

for i in range (arr_size * z):
    y_arr_misaligned.append(1)
    
y_arr_misaligned = np.array(y_arr_misaligned)

In [19]:
print(y_arr_misaligned)
print(len(y_arr_misaligned))

[1 1 1 ... 1 1 1]
1200000


### state_arr_misaligned

In [20]:
state_arr_misaligned = []

for i in range (arr_size):
    for j in range (z):
        state_arr_misaligned.append(state_arr[i])
        
state_arr_misaligned = np.array(state_arr_misaligned)

In [21]:
print(state_arr_misaligned)

[[-1.         -1.         -1.        ]
 [-1.         -1.         -1.        ]
 [-1.         -1.         -1.        ]
 ...
 [ 6.6821086   6.26530121 13.77949172]
 [ 6.6821086   6.26530121 13.77949172]
 [ 6.6821086   6.26530121 13.77949172]]


In [22]:
print(len(state_arr_misaligned))

1200000


### explanation_arr_misaligned

In [23]:
explanation_arr_misaligned = []

explanations = ['impossible matching',
 'long distance travelled and high CO2 emissions',
 'long distance travelled and long distance travelled with no freight',
 'long distance travelled with no freight and high CO2 emissions',
 'long distance travelled, long distance travelled with no freight, and high CO2 emissions',
 'short distance travelled and low CO2 emissions',
 'short distance travelled and short distance travelled with no freight',
 'short distance travelled with no freight and low CO2 emissions',
 'short distance travelled, short distance travelled with no freight, and low CO2 emissions']

for i in range (arr_size):
    current_explanation = explanation_arr[i]
    for explanation in explanations:
        if (explanation != current_explanation):
            explanation_arr_misaligned.append(explanation)
            
explanation_arr_misaligned = np.array(explanation_arr_misaligned)

In [24]:
print(explanation_arr_misaligned)
print(len(explanation_arr_misaligned))

['long distance travelled and high CO2 emissions'
 'long distance travelled and long distance travelled with no freight'
 'long distance travelled with no freight and high CO2 emissions' ...
 'short distance travelled and short distance travelled with no freight'
 'short distance travelled with no freight and low CO2 emissions'
 'short distance travelled, short distance travelled with no freight, and low CO2 emissions']
1200000


In [25]:
unique, counts = np.unique(explanation_arr_misaligned, return_counts=True)
dict(zip(unique, counts))

{'impossible matching': 130000,
 'long distance travelled and high CO2 emissions': 140663,
 'long distance travelled and long distance travelled with no freight': 121782,
 'long distance travelled with no freight and high CO2 emissions': 149004,
 'long distance travelled, long distance travelled with no freight, and high CO2 emissions': 120105,
 'short distance travelled and low CO2 emissions': 134737,
 'short distance travelled and short distance travelled with no freight': 138117,
 'short distance travelled with no freight and low CO2 emissions': 147475,
 'short distance travelled, short distance travelled with no freight, and low CO2 emissions': 118117}

## Join the Datasets

In [26]:
new_y_arr = np.concatenate((y_arr, y_arr_misaligned))
print(new_y_arr)
print(len(new_y_arr))

[0 0 0 ... 1 1 1]
1350000


In [27]:
new_state_arr = np.concatenate((state_arr, state_arr_misaligned))
print(new_state_arr)
print(len(new_state_arr))

[[-1.         -1.         -1.        ]
 [-1.         -1.         -1.        ]
 [-1.         -1.         -1.        ]
 ...
 [ 6.6821086   6.26530121 13.77949172]
 [ 6.6821086   6.26530121 13.77949172]
 [ 6.6821086   6.26530121 13.77949172]]
1350000


In [28]:
new_explanation_arr = np.concatenate((explanation_arr, explanation_arr_misaligned))
print(new_explanation_arr)
print(len(new_explanation_arr))

['impossible matching' 'impossible matching' 'impossible matching' ...
 'short distance travelled and short distance travelled with no freight'
 'short distance travelled with no freight and low CO2 emissions'
 'short distance travelled, short distance travelled with no freight, and low CO2 emissions']
1350000


## Shuffle the Data

In [29]:
#from https://stackoverflow.com/questions/4601373/better-way-to-shuffle-two-numpy-arrays-in-unison
def unison_shuffled_copies(a, b, c):
    assert len(a) == len(b) == len(c)
    p = np.random.permutation(len(a))
    return a[p], b[p], c[p]

In [30]:
final_y_arr, final_state_arr, final_explanation_arr = unison_shuffled_copies(new_y_arr, new_state_arr, new_explanation_arr)

In [31]:
print(final_y_arr)
print(len(final_y_arr))

[1 1 1 ... 1 1 1]
1350000


In [32]:
print(final_state_arr)
print(len(final_state_arr))

[[ 6.8956827   6.20253552 13.71397357]
 [ 6.28039584  5.32787617 14.71285031]
 [ 5.7365723   5.32300998 13.54302804]
 ...
 [ 6.16751649  5.6094718  13.25830992]
 [ 5.7401144   5.42934563 11.9253547 ]
 [ 6.36475076  4.96284463 14.63028117]]
1350000


In [33]:
print(final_explanation_arr)
print(len(final_explanation_arr))

['short distance travelled and low CO2 emissions'
 'short distance travelled with no freight and low CO2 emissions'
 'short distance travelled and short distance travelled with no freight'
 ... 'short distance travelled and low CO2 emissions'
 'long distance travelled, long distance travelled with no freight, and high CO2 emissions'
 'short distance travelled and low CO2 emissions']
1350000


## Save the final dataset

In [34]:
with open('datasets/data/y_arr.npy', 'wb') as f:
    np.save(f, final_y_arr)
    
with open('datasets/data/state.npy', 'wb') as f:
    np.save(f, final_state_arr)
    
with open('datasets/data/explanation.npy', 'wb') as f:
    np.save(f, final_explanation_arr)